# 01 - Exploratory Data Analysis (EDA)

## Overview

Load and explore the CICIoT2023 dataset: shape, columns, data types, missing values, and label distribution.

**Dataset location**: `/home/willtanoe/Documents/CICIOT23`

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Navigate to project root (one level up from notebooks/)
os.chdir(Path.cwd().parent)

# Paths
DATA_DIR = "dataset-raw"
TRAIN_CSV = os.path.join(DATA_DIR, "train", "train.csv")
VAL_CSV   = os.path.join(DATA_DIR, "validation", "validation.csv")
TEST_CSV  = os.path.join(DATA_DIR, "test", "test.csv")

print(f"Working directory: {os.getcwd()}")
print(f"Dataset directory: {DATA_DIR}")
print(f"Train: {TRAIN_CSV}")
print(f"Validation: {VAL_CSV}")
print(f"Test: {TEST_CSV}")

## 1. Load Data

In [ ]:
# Load training data (the largest)
print("Loading train.csv...")
train = pd.read_csv(TRAIN_CSV)
print(f"Train shape: {train.shape}")

print("\nLoading validation.csv...")
val = pd.read_csv(VAL_CSV)
print(f"Validation shape: {val.shape}")

print("\nLoading test.csv...")
test = pd.read_csv(TEST_CSV)
print(f"Test shape: {test.shape}")

## 2. Basic Info

In [ ]:
print("=" * 60)
print("TRAINING DATA INFO")
print("=" * 60)
print(f"\nShape: {train.shape[0]:,} rows × {train.shape[1]} columns")
print(f"\nColumn names:\n{train.columns.tolist()}")
print(f"\nData types:\n{train.dtypes.value_counts()}")

## 3. Missing Values

In [ ]:
print("\nMissing values in train:")
missing = train.isnull().sum()
missing_pct = (missing / len(train) * 100).round(2)
missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
}).sort_values('missing_count', ascending=False)

if missing.sum() == 0:
    print("No missing values found!")
else:
    print(missing_df[missing_df['missing_count'] > 0])

print(f"\nMissing values in val: {val.isnull().sum().sum()}")
print(f"Missing values in test: {test.isnull().sum().sum()}")

## 4. Label Distribution

In [ ]:
print("\n" + "=" * 60)
print("LABEL DISTRIBUTION (Train)")
print("=" * 60)

label_counts = train['label'].value_counts()
label_pct = (label_counts / len(train) * 100).round(2)

label_df = pd.DataFrame({
    'count': label_counts,
    'percentage': label_pct
})
print(label_df)

print(f"\nTotal unique labels: {train['label'].nunique()}")

In [ ]:
# Plot label distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Bar plot
ax1 = axes[0]
label_counts.plot(kind='bar', ax=ax1, color='steelblue')
ax1.set_title('Label Distribution (Count)', fontsize=14)
ax1.set_xlabel('Label')
ax1.set_ylabel('Count')
ax1.tick_params(axis='x', rotation=90)

# Log scale bar plot
ax2 = axes[1]
label_counts.plot(kind='bar', ax=ax2, color='coral')
ax2.set_title('Label Distribution (Log Scale)', fontsize=14)
ax2.set_xlabel('Label')
ax2.set_ylabel('Count (log)')
ax2.set_yscale('log')
ax2.tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.savefig('results/figures/01_label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFigure saved: results/figures/01_label_distribution.png")

## 5. Statistical Summary

In [ ]:
print("\n" + "=" * 60)
print("STATISTICAL SUMMARY (Numeric Features)")
print("=" * 60)

desc = train.describe().T
desc['missing'] = train.isnull().sum()
desc['unique'] = train.nunique()
print(desc)

In [ ]:
# Check for infinite values
print("\n" + "=" * 60)
print("CHECKING FOR INFINITE VALUES")
print("=" * 60)

numeric_cols = train.select_dtypes(include=[np.number]).columns
inf_counts = train[numeric_cols].apply(lambda x: np.isinf(x).sum())
inf_df = inf_counts[inf_counts > 0]

if len(inf_df) == 0:
    print("No infinite values found.")
else:
    print(f"Columns with infinite values:\n{inf_df}")

# Check for constant columns (zero variance)
print("\nConstant columns (zero variance):")
const_cols = train[numeric_cols].columns[train[numeric_cols].std() == 0].tolist()
if const_cols:
    print(const_cols)
else:
    print("None")

## 6. Class Imbalance Analysis

In [ ]:
print("\n" + "=" * 60)
print("CLASS IMBALANCE ANALYSIS")
print("=" * 60)

counts = train['label'].value_counts()
max_count = counts.max()
min_count = counts.min()
imbalance_ratio = max_count / min_count

print(f"Most frequent class: {counts.index[0]} ({counts.iloc[0]:,} samples, {counts.iloc[0]/len(train)*100:.2f}%)")
print(f"Least frequent class: {counts.index[-1]} ({counts.iloc[-1]:,} samples, {counts.iloc[-1]/len(train)*100:.4f}%)")
print(f"\nImbalance ratio (max/min): {imbalance_ratio:,.0f}x")

# Calculate minority class threshold
minority_threshold = 0.1  # classes with less than 0.1% of data
minority_count = (label_pct < minority_threshold).sum()
print(f"\nMinority classes (<0.1%): {minority_count}")

# Imbalance categories
print("\nClass size categories:")
print(f"  > 10%: {sum(label_pct > 10)}")
print(f"  1-10%: {sum((label_pct > 1) & (label_pct <= 10))}")
print(f"  0.1-1%: {sum((label_pct > 0.1) & (label_pct <= 1))}")
print(f"  < 0.1%: {sum(label_pct <= 0.1)}")

## 7. Feature Correlation (Top Features)

In [ ]:
# Correlation heatmap for top 15 features
numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()

# Get top features by variance
variances = train[numeric_cols].var().sort_values(ascending=False)
top_features = variances.head(15).index.tolist()

fig, ax = plt.subplots(figsize=(12, 10))
corr = train[top_features].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Heatmap (Top 15 by Variance)', fontsize=14)
plt.tight_layout()
plt.savefig('results/figures/01_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFigure saved: results/figures/01_correlation_heatmap.png")

## 8. Save Summary Stats

In [ ]:
# Save label distribution to CSV
label_df.to_csv('results/logs/01_label_distribution.csv', index=True)
print("Label distribution saved: results/logs/01_label_distribution.csv")

# Save summary to JSON
import json

summary = {
    'train_rows': int(len(train)),
    'val_rows': int(len(val)),
    'test_rows': int(len(test)),
    'num_features': int(train.shape[1] - 1),
    'num_classes': int(train['label'].nunique()),
    'missing_values': int(train.isnull().sum().sum()),
    'imbalance_ratio': float(imbalance_ratio),
    'constant_columns': const_cols,
    'top_features_by_variance': top_features[:10]
}

with open('results/logs/01_eda_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print("EDA summary saved: results/logs/01_eda_summary.json")

## Summary

- **Training**: 5.49M rows × 47 columns
- **Validation/Test**: 1.18M rows each
- **Features**: 46 numeric features + 1 label column
- **Classes**: 34 (highly imbalanced, ratio ~6,058x)
- **Missing values**: None
- **Infinite values**: Checked, none found

**Next**: Proceed to `02_preprocessing.ipynb` for data cleaning and preparation.